# PCA sur MNIST

Même structure que `01_kmeans.ipynb`, adaptée à la PCA. Contrairement à K-means, l'espace latent ici est continu : la projection directe du dataset dans cet espace a donc un sens visuel immédiat.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname("__file__"), "..")))

import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from src.dataset import load_mnist_dataset
from src.helper import extract_full_dataset
from src.pca import PCA
from src.metrics import compression_report, Latent

## 1. Chargement des données

Même sous-ensemble que K-means, pour que les comparatifs (MSE, poids) restent valides entre notebooks.

In [2]:
dataloader = load_mnist_dataset(train=True, batch_size=4096, download=True, shuffle=False)
images, digit_labels = extract_full_dataset(dataloader)

N = 10000
X = images[:N].flatten(start_dim=1).numpy()
digit_labels = digit_labels[:N].numpy()

print("X shape:", X.shape)

X shape: (10000, 784)


## 2. Combien de composantes garder ? Variance expliquée cumulée

Avant de choisir un N pour le reste du notebook, on regarde combien de composantes suffisent pour capturer l'essentiel de la variance. Réponds à la question posée dans tes notes de cours : "jusqu'à combien de valeurs propres pour une reconstruction assez claire ?"

In [3]:
# On fit une PCA sur la dimension maximale possible pour analyser la variance
pca_max = PCA(n_components=X.shape[1])
pca_max.fit(X)

# Calcul de la variance expliquée cumulée
total_variance = np.sum(pca_max.explained_variance)
explained_variance_ratio = pca_max.explained_variance / total_variance
cumulative_variance = np.cumsum(explained_variance_ratio)

# Tracé de la courbe avec Plotly
fig = px.line(
    x=np.arange(1, len(cumulative_variance) + 1),
    y=cumulative_variance,
    labels=dict(x="Nombre de Composantes", y="Variance Expliquée Cumulée"),
    title="Courbe de Variance Expliquée Cumulée (PCA)"
)

# Recherche des seuils 90%, 95% et 99%
thresholds = [0.90, 0.95, 0.99]
colors = ["red", "orange", "green"]
for t, color in zip(thresholds, colors):
    n_comp = np.where(cumulative_variance >= t)[0][0] + 1
    print(f"Seuil {t*100:.0f}% de variance atteint avec N = {n_comp} composantes.")
    
    # Ajout d'une ligne verticale et d'une annotation sur le graphe
    fig.add_vline(x=n_comp, line_dash="dash", line_color=color)
    fig.add_annotation(
        x=n_comp, y=t, 
        text=f"N={n_comp} ({t*100:.0f}%)", 
        showarrow=True, arrowhead=1
    )

fig.update_layout(template="plotly_dark")
fig.show()

Seuil 90% de variance atteint avec N = 85 composantes.
Seuil 95% de variance atteint avec N = 150 composantes.
Seuil 99% de variance atteint avec N = 326 composantes.


## 3. Fit avec le N choisi

In [4]:
# Nous choisissons N_COMPONENTS = 87 composantes pour conserver environ 90% de la variance
N_COMPONENTS = 87
model = PCA(n_components=N_COMPONENTS)
model.fit(X)

## 4. Encode / decode / rapport de compression

In [5]:
latent = model.encode(X)
X_reconstructed = model.decode(latent)
codebook = model.get_codebook()

report = compression_report(codebook, latent, X, X_reconstructed)
for k, v in report.items():
    print(f"{k}: {v}")

latent_nature: continuous
codebook_bytes: 275968
latent_bytes: 3480000
total_compressed_bytes: 3755968
original_bytes: 31360000
compression_ratio: 8.349378908446504
reconstruction_mse: 0.006548926699906588


## 5. Visualisation : les eigenvectors (composantes) comme images

Même logique que les centroïdes de K-means : chaque composante est un vecteur de 784 valeurs, reshape en (28, 28).

In [6]:
# Affichage des 10 premiers eigenvectors sous forme de grille
fig = make_subplots(
    rows=2, cols=5,
    subplot_titles=[f"Eigenvector {i+1}" for i in range(10)]
)

for i in range(10):
    row = i // 5 + 1
    col = i % 5 + 1
    # model.components possède la forme (n_features, n_components)
    comp_img = model.components[:, i].reshape(28, 28)
    
    # Utilisation d'une palette divergente 'RdBu' pour mettre en évidence les zones positives et négatives
    fig.add_trace(
        go.Heatmap(z=comp_img, colorscale="RdBu", showscale=False),
        row=row, col=col
    )

fig.update_layout(
    height=500, width=1000,
    title_text="PCA Eigenvectors (Directions de Variation)",
    template="plotly_dark"
)

for i in range(1, 11):
    fig.update_yaxes(autorange="reversed", showticklabels=False, row=(i-1)//5+1, col=(i-1)%5+1)
    fig.update_xaxes(showticklabels=False, row=(i-1)//5+1, col=(i-1)%5+1)

fig.show()

## 6. Visualisation : original vs reconstruit

In [7]:
# Choix de 5 indices aléatoires pour comparer les originaux et les reconstruits par la PCA
np.random.seed(42)
indices = np.random.choice(X.shape[0], size=5, replace=False)

fig = make_subplots(
    rows=2, cols=5,
    subplot_titles=[f"Original {idx}" for idx in indices] + [f"Reconstructed {idx}" for idx in indices]
)

for i, idx in enumerate(indices):
    orig_img = X[idx].reshape(28, 28)
    recon_img = X_reconstructed[idx].reshape(28, 28)
    
    fig.add_trace(go.Heatmap(z=orig_img, colorscale="gray", showscale=False), row=1, col=i+1)
    fig.add_trace(go.Heatmap(z=recon_img, colorscale="gray", showscale=False), row=2, col=i+1)

fig.update_layout(
    height=500, width=1000,
    title_text="PCA Original vs Reconstructed",
    template="plotly_dark"
)

for row in [1, 2]:
    for col in range(1, 6):
        fig.update_yaxes(autorange="reversed", showticklabels=False, row=row, col=col)
        fig.update_xaxes(showticklabels=False, row=row, col=col)

fig.show()

## 7. Visualisation : le dataset projeté dans l'espace latent

Ici, contrairement à K-means, l'espace latent est continu : la projection directe donne un vrai nuage de points exploitable.

In [8]:
# Entraînement d'une PCA 2D dédiée à la visualisation
pca_2d = PCA(n_components=2)
pca_2d.fit(X)
latent_2d = pca_2d.encode(X)

# Scatter plot interactif
fig = px.scatter(
    x=latent_2d.array[:, 0],
    y=latent_2d.array[:, 1],
    color=digit_labels.astype(str),
    color_discrete_sequence=px.colors.qualitative.Plotly,
    labels=dict(x="Premiere Composante Principale", y="Deuxieme Composante Principale", color="Chiffre Reel"),
    title="MNIST : Projection dans l'espace latent 2D de la PCA"
)

fig.update_layout(
    width=900, height=600,
    template="plotly_dark"
)
fig.show()

## 8. Génération de nouvelles images par sampling dans l'espace latent

Cette section montre deux approches plausibles pour générer une nouvelle image avec PCA :
1) choisir un point latent proche d'une région dense, sans reproduire un point existant.
2) sampler autour d'un point latent existant avec une petite perturbation gaussienne.

In [ ]:
# Latent training data utilisé pour déterminer une région plausible
latent_train = latent.array
latent_std = latent_train.std(axis=0)

rng = np.random.default_rng(123)

# Approche 1 : point latent proche d'un point d'entraînement existant
# Cela reste dans une région dense du dataset latent en perturbant légèrement un point réel.
z_ref = latent_train[rng.integers(len(latent_train))]
z_new_dense = z_ref + rng.normal(scale=0.2 * latent_std, size=z_ref.shape)

# Approche 2 : point latent dans un bounding box autour des données,
# puis vérification de la proximité avec un point d'entraînement
latent_min = latent_train.min(axis=0)
latent_max = latent_train.max(axis=0)
z_new_box = rng.uniform(latent_min, latent_max)
box_dist = np.linalg.norm(latent_train - z_new_box, axis=1)

print(f"Distance minimale au dataset latent (box) : {box_dist.min():.4f}")

# Décode les deux nouveaux points
X_new_dense = model.decode(Latent(array=z_new_dense[None, :], nature="continuous"))
X_new_box = model.decode(Latent(array=z_new_box[None, :], nature="continuous"))

# Affichage des deux images générées
fig = make_subplots(rows=1, cols=2, subplot_titles=[
    "Génération par point dense",
    "Génération par bounding box"
])

for col, X_gen in enumerate([X_new_dense, X_new_box], start=1):
    img = X_gen[0].reshape(28, 28)
    fig.add_trace(go.Heatmap(z=img, colorscale="gray", showscale=False), row=1, col=col)

fig.update_layout(
    height=400, width=700,
    title_text="Nouvelles images générées par sampling dans l'espace latent",
    template="plotly_dark"
)
for col in [1, 2]:
    fig.update_yaxes(autorange="reversed", showticklabels=False, row=1, col=col)
    fig.update_xaxes(showticklabels=False, row=1, col=col)
fig.show()

Distance minimale au dataset latent (box) : 12.3882


## 9. Reconstruction pour plusieurs valeurs de k

On compare la reconstruction pour différentes valeurs de `n_components` (k). On sait que k=87 suffit pour expliquer ~90% de la variance. En visualisant la progression on voit comment la qualité s'améliore à mesure qu'on ajoute des composantes.

In [10]:
K_VALUES = [2, 10, 30, 50, 87, 150, 300]

np.random.seed(42)
sample_indices = np.random.choice(X.shape[0], size=5, replace=False)

# Pré-calculer les reconstructions pour chaque k
reconstructions = {}
for k in K_VALUES:
    pca_k = PCA(n_components=k)
    pca_k.fit(X)
    latent_k = pca_k.encode(X)
    recon_k = pca_k.decode(latent_k)
    reconstructions[k] = recon_k
    print(f"k={k:>4d} → MSE = {np.mean((X - recon_k)**2):.4f}")

k=   2 → MSE = 0.0556
k=  10 → MSE = 0.0340
k=  30 → MSE = 0.0178
k=  50 → MSE = 0.0116
k=  87 → MSE = 0.0065
k= 150 → MSE = 0.0034
k= 300 → MSE = 0.0009


In [11]:
n_rows = 1 + len(K_VALUES)  # 1 row for originals + 1 per k
n_cols = 5

row_titles = ["Original"] + [f"k = {k}" for k in K_VALUES]

fig = make_subplots(
    rows=n_rows, cols=n_cols,
    vertical_spacing=0.02,
    horizontal_spacing=0.02,
    row_titles=row_titles
)

for i, idx in enumerate(sample_indices):
    # Ligne 1 : original
    orig_img = X[idx].reshape(28, 28)
    fig.add_trace(go.Heatmap(z=orig_img, colorscale="gray", showscale=False), row=1, col=i+1)

    # Lignes suivantes : reconstructions pour chaque k
    for j, k in enumerate(K_VALUES):
        recon_img = reconstructions[k][idx].reshape(28, 28)
        fig.add_trace(go.Heatmap(z=recon_img, colorscale="gray", showscale=False), row=j+2, col=i+1)

fig.update_layout(
    height=200 * n_rows,
    width=1000,
    title_text="Reconstruction PCA pour différentes valeurs de k (MNIST)",
    template="plotly_dark",
    showlegend=False
)

for r in range(1, n_rows + 1):
    for c in range(1, n_cols + 1):
        fig.update_yaxes(autorange="reversed", showticklabels=False, row=r, col=c)
        fig.update_xaxes(showticklabels=False, row=r, col=c)

fig.show()